<a href="https://colab.research.google.com/github/YmerinAmout-code/colab-notebooks/blob/main/projetH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors flask
!pip install -q huggingface_hub omegaconf
!npm install -g localtunnel > /dev/null 2>&1
print('✅ Dependencies installed')

✅ Dependencies installed


In [2]:
# Cell 2: Model definitions + lazy loader
import torch, gc
from diffusers import (
    StableDiffusionXLPipeline,
    StableDiffusionPipeline,
    EulerAncestralDiscreteScheduler
)

# FIX: unified keys matching frontend (pony/illustrious/aom3/meinamix)
# FIX: type uses 'sdxl' / 'sd15' (matches frontend RESOLUTIONS dict)
# FIX: neg_default added for each model
# FIX: AOM3 uses from_single_file (url key instead of id)
MODELS = {
    'pony': {
        'id': 'cagliostrolab/animagine-xl-3.1',
        'type': 'sdxl',
        'label': 'Animagine XL 3.1',
        'neg_default': 'ugly, deformed, blurry, low quality, bad anatomy, text, watermark, poorly drawn, signature'
    },
    'illustrious': {
        'id': 'Eugeoter/illustrious-xl-v0.1-pruned',
        'type': 'sdxl',
        'label': 'Illustrious XL',
        'neg_default': 'ugly, deformed, blurry, low quality, bad anatomy, text, watermark, extra limbs'
    },
    'aom3': {
        'url': 'https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AbyssOrangeMix3_hard.safetensors',
        'type': 'sd15',
        'label': 'AOM3 (AbyssOrangeMix3)',
        'neg_default': '(worst quality, low quality:1.4), (zombie, sketch, interlocked fingers, comic):1.1, fat, obese, chubby, deformed, blurry, bad anatomy, disfigured, mutation, extra_limb, ugly, messy drawing'
    },
    'meinamix': {
        'id': 'Meina/MeinaMix_V11',
        'type': 'sd15',
        'label': 'MeinaMix V11',
        'neg_default': '(worst quality, low quality:1.4), (zombie, sketch, interlocked fingers, comic):1.1, lowres, normal quality, skin spots, acnes, skin blemishes, age spot, ugly, duplicate, morbid'
    }
}

current_model = None
current_pipe = None

def load_model(model_key):
    global current_model, current_pipe
    if model_key == current_model and current_pipe is not None:
        return current_pipe
    if model_key not in MODELS:
        raise ValueError(f'Unknown model: {model_key}')

    # Free previous model from GPU (lazy loading — one model at a time)
    if current_pipe is not None:
        del current_pipe
        current_pipe = None
        gc.collect()
        torch.cuda.empty_cache()

    info = MODELS[model_key]
    print(f'Loading {info["label"]}...')

    if info['type'] == 'sdxl':
        # FIX: from_pretrained for SDXL models
        # FIX: VAE kept in fp32 for SDXL stability (avoids NaN artifacts)
        pipe = StableDiffusionXLPipeline.from_pretrained(
            info['id'],
            torch_dtype=torch.float16,
            use_safetensors=True,
        )
        pipe.vae.to(dtype=torch.float32)  # VAE fp32, rest fp16

    elif 'url' in info:
        # FIX: AOM3 uses from_single_file (safetensors checkpoint)
        pipe = StableDiffusionPipeline.from_single_file(
            info['url'],
            torch_dtype=torch.float16,
            safety_checker=None,
            requires_safety_checker=False
        )

    else:
        # SD 1.5 via from_pretrained (MeinaMix)
        pipe = StableDiffusionPipeline.from_pretrained(
            info['id'],
            torch_dtype=torch.float16,
            safety_checker=None,
            requires_safety_checker=False
        )

    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to('cuda')
    pipe.enable_attention_slicing()

    current_model = model_key
    current_pipe = pipe
    print(f'  {info["label"]} loaded on GPU!')
    return pipe

# Load default model (Pony V6 XL)
load_model('pony')
print('\n Multi-model system ready!')
print('Available:', list(MODELS.keys()))


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading Pony Diffusion V6 XL...


Couldn't connect to the Hub: 404 Client Error. (Request ID: Root=1-69a73b0f-1af3d8d462d18b1566c2122e;f6649223-0aa8-47c6-9cf5-5df7fbf57788)

Repository Not Found for url: https://huggingface.co/api/models/John6666/pony-diffusion-v6-xl-sdxl.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication.
Will try to load from local cache.


OSError: Cannot load model John6666/pony-diffusion-v6-xl-sdxl: model is not cached locally and an error occurred while trying to fetch metadata from the Hub. Please check out the root cause in the stacktrace above.

In [ ]:
# Cell 3: Flask API server + localtunnel
from flask import Flask, request, send_file, jsonify, Response
import io, subprocess, threading, time

app = Flask(__name__)

@app.after_request
def add_cors(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = '*'
    response.headers['Access-Control-Allow-Methods'] = '*'
    return response

@app.route('/health')
def health():
    # FIX: return 'model_name' field (frontend reads info.model_name)
    return jsonify({
        'status': 'ok',
        'model': current_model,
        'model_name': MODELS.get(current_model, {}).get('label', 'unknown')
    })

@app.route('/models', methods=['GET'])
def list_models():
    # FIX: return dict keyed by model key (frontend does modelsInfo[key].type / .neg_default)
    # was: returning a list — broke modelsInfo[currentModel] lookups
    models_dict = {}
    for key, info in MODELS.items():
        models_dict[key] = {
            'label': info['label'],
            'type': info['type'],
            'neg_default': info.get('neg_default', '')
        }
    return jsonify({'models': models_dict, 'current': current_model})

@app.route('/switch', methods=['POST', 'OPTIONS'])
def switch_model():
    if request.method == 'OPTIONS':
        return Response(status=200)
    data = request.json or {}
    model_key = data.get('model', 'pony')
    if model_key not in MODELS:
        return jsonify({'ok': False, 'error': f'Unknown model: {model_key}'}), 400
    try:
        load_model(model_key)
        # FIX: return {ok: True, name: ...} — frontend checks data.ok and data.name
        # was: {status:'ok', label:...} — data.ok was undefined (falsy) so switch always failed
        return jsonify({'ok': True, 'model': model_key, 'name': MODELS[model_key]['label']})
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/generate', methods=['POST', 'OPTIONS'])
def generate():
    if request.method == 'OPTIONS':
        return Response(status=200)
    data = request.json or {}
    prompt = data.get('prompt', '').strip()
    if not prompt:
        return jsonify({'error': 'prompt is required'}), 400

    negative = data.get('negative_prompt', '') or MODELS.get(current_model, {}).get('neg_default', 'ugly, deformed, blurry, low quality')
    steps = max(10, min(int(data.get('steps', 25)), 50))
    # FIX: cfg_scale now used from request (was hardcoded 7.5)
    cfg_scale = float(data.get('cfg_scale', 7.0))

    # Smart defaults based on model type
    model_type = MODELS.get(current_model, {}).get('type', 'sd15')
    default_size = 1024 if model_type == 'sdxl' else 512
    width = int(data.get('width', default_size))
    height = int(data.get('height', default_size))

    # Enrich prompt
    full_prompt = f'{prompt}, masterpiece, best quality, highly detailed'

    try:
        pipe = current_pipe
        if pipe is None:
            return jsonify({'error': 'No model loaded'}), 503
        image = pipe(
            full_prompt,
            negative_prompt=negative,
            num_inference_steps=steps,
            guidance_scale=cfg_scale,
            width=width,
            height=height
        ).images[0]
        buf = io.BytesIO()
        image.save(buf, format='PNG')
        buf.seek(0)
        return send_file(buf, mimetype='image/png')
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Start Flask in background thread
def run_flask():
    app.run(port=5000, host='0.0.0.0', use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

# Start localtunnel
proc = subprocess.Popen(['lt', '--port', '5000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
time.sleep(5)
line = proc.stdout.readline().strip()
print(f'\n ProjetH API is live!')
print(f'URL: {line}')
print(f'Current model: {MODELS[current_model]["label"]}')
print(f'\nEndpoints:')
print(f'  GET  /health          - Status + model_name')
print(f'  GET  /models          - Dict of all models with type + neg_default')
print(f'  POST /switch          - Switch model {{"model": "pony"|"illustrious"|"aom3"|"meinamix"}}')
print(f'  POST /generate        - Generate image')
print(f'\nCopy the URL above into the web app backend field!')

# Keep-alive
while True:
    time.sleep(60)
